<a href="https://colab.research.google.com/github/Abdul-RahmanAI/urdu-ocr-codesaviours-si26--AbdulRahman-/blob/main/SI26_Week3_%5BAbdulRahman%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 – Building a Custom PyTorch Dataset Class for Urdu OCR

## Objective

The objective of this notebook is to create a custom PyTorch Dataset class for loading the Kaggle Urdu OCR dataset. The Dataset class reads the image paths and corresponding Urdu text labels from `labels.csv`, processes the images using the TrOCR processor, and prepares the data for deep learning model training.



In [14]:
import os
print(os.listdir("/content"))

['.config', 'realgt', 'urdu_ocr_data.rar', 'labels.csv', '.ipynb_checkpoints', 'line_undegraded', 'sample_data']


In [15]:
!apt-get install -q unrar
!unrar x "/content/urdu_ocr_data.rar" "/content/"

Streaming output truncated to the last 5000 lines.
Extracting  /content/line_undegraded/line_undegraded/55.png               52%  OK 
Extracting  /content/line_undegraded/line_undegraded/550.png              52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5500.png             52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5501.png             52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5502.png             52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5503.png             52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5504.png             52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5505.png             52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5506.png             52%  OK 
Extracting  /content/line_undegraded/line_undegraded/5507.png             52%  OK 
Extracting  /content/line_und

In [16]:
import os

print(os.listdir("/content"))

['.config', 'realgt', 'urdu_ocr_data.rar', 'labels.csv', '.ipynb_checkpoints', 'line_undegraded', 'sample_data']


In [17]:
# Install required libraries
!pip install -q transformers==4.38.2
!pip install -q tokenizers==0.15.2
!pip install -q sentencepiece
!pip install -q tiktoken
!pip install -q torch pillow pandas

import os
import torch
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd


class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, image_folder, processor):

        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.image_folder = image_folder

        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        # Image name stored in labels.csv
        image_name = row["image"].lstrip("/")

        # Complete image path
        image_path = os.path.join(self.image_folder, image_name)

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Process image
        encoding = self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values = encoding.pixel_values.squeeze()

        # Tokenize Urdu text
        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            truncation=True,
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

print("Dataset class created successfully!")

Dataset class created successfully!


In [18]:
# Load TrOCR processor
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

# Correct image folder
image_folder = "/content/line_undegraded/line_undegraded"

# Create dataset
dataset = UrduOCRDataset(
    csv_path="/content/labels.csv",
    image_folder=image_folder,
    processor=processor
)

# Test first sample
sample = dataset[0]

print("Pixel Values Shape:", sample["pixel_values"].shape)
print("Labels Shape:", sample["labels"].shape)

print("\nDataset is working correctly!")

# Train-Test Split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f"Training Samples: {train_size}")
print(f"Testing Samples: {test_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Dataset loaded: 10063 samples
Pixel Values Shape: torch.Size([3, 384, 384])
Labels Shape: torch.Size([128])

Dataset is working correctly!
Training Samples: 8050
Testing Samples: 2013


## Confirmation

My dataset has **10,064 images** and loads correctly.

The custom `UrduOCRDataset` class successfully loads the images and their corresponding text labels. The dataset was successfully split into training and testing sets and is ready for model training.